# Harmonic fields and harmonic vector potentials on a hollow torus

A *hollow torus* is a torus with a concentric toroidal cavity removed. Topologically it has

* **one cavity** (its boundary has two connected components), hence a one-dimensional space of
  *normal* harmonic fields $\mathcal{H}^1_0 = \{A \in H_0(curl) : curl\,A = 0,\ \widetilde{div}\,A = 0\}$;
* **two tunnels** (a toroidal and a poloidal one), hence a two-dimensional space of
  *tangent* harmonic fields $\mathcal{H}^2 = \{B \in H_0(div) : div\,B = 0,\ \widetilde{curl}\,B = 0\}$.

In this notebook we compute, with the PSYDAC FEEC tools,

1. a basis of the normal harmonic fields, as the kernel of the discrete 1-Hodge-Laplacian with homogeneous boundary conditions;
2. two harmonic vector potentials $A_{H,i} \in H(curl)$ whose curls $B_{H,i} = curl\,A_{H,i}$ span the tangent harmonic fields.

The construction of the harmonic vector potentials follows M. Campos Pinto and J. Owezarek
(https://arxiv.org/abs/2508.16822): each potential is split as $A_{H,i} = A^b_{H,i} + A^0_{H,i}$
into a *lifting* potential $A^b_{H,i} \in H(curl)$ and a *correction* potential $A^0_{H,i} \in H_0(curl)$.

The notebook is the interactive version of the script `examples/feec/feec_potential_hollowtorus.py`.

## Step 0 : Parameters

The domain is parametrized on the unit cube $(0,1)^3$ with logical coordinates
(normalized radius, poloidal angle, toroidal angle), hence periodic in the last two directions.

Several mappings are available:

* `'hollow_torus'` : analytic `HollowTorusUnit` mapping (default, no extra dependency);
* `'str_hollow_torus'` : the Struphy `HollowTorus` domain, interpolated by a spline mapping;
* `'hollow_desc'`, `'hollow_gvec'` : Struphy DESC / GVEC equilibria with a cavity, interpolated by a spline mapping.

The Struphy mappings require `struphy` (and the corresponding equilibrium codes) to be installed.

In [ ]:
import os
import sys
import time

import numpy as np

# utils_harpo.py (hollow torus mappings, eigenvalue solver, Struphy mapping wrapper) lives in examples/feec
sys.path.insert(0, os.path.abspath(os.path.join('..', 'feec')))

# ----- Domain parameters -----
r      = 1.            # inner radius of the torus
R      = 3.            # outer radius of the torus
minRad = (R - r) / 2   # minor radius (radius of the tube)
majRad = (R + r) / 2   # major radius (center of the tube)
cavRad = 0.5           # radius of the toroidal cavity (same major radius as the torus)
assert cavRad < minRad, "the cavity must be smaller than the torus itself"

mapping_name = 'hollow_desc'   # 'hollow_torus', 'str_hollow_torus', 'hollow_desc' or 'hollow_gvec'

# ----- Discretization parameters -----
ncells   = [4, 4, 4]            # number of cells in each logical direction
degree   = [2, 2, 2]            # B-spline degree in each logical direction
periodic = [False, True, True]  # periodicity of the logical domain

# Tolerance of the correction potential solver - defines the "accuracy" of the harmonic vector potentials
cor_tol = 1e-12

# ----- Output -----
save        = True
output_dir  = 'hollow_torus_output'
params_name = f'visu_{mapping_name}_cavRad{cavRad}_n{ncells[0]}_{ncells[1]}_{ncells[2]}_d{degree[0]}_tol{cor_tol}'
vtu_file    = f'{output_dir}/{params_name}.static.vtu'

## Step 1 : Domain and de Rham sequence

### Define the mapped domain

For the analytic mapping the symbolic domain is directly the image of the logical cube.
For the Struphy mappings, the Struphy domain is wrapped in a `BasicCallableMapping`
(`StruphyCallableMapping`), interpolated by a `SplineCallableMapping`, and turned into a symbolic
`DefinedMapping` whose callable is the spline itself.

In [20]:
from sympde.topology         import Cube
from psydac.mapping.discrete import SplineCallableMapping

from utils_harpo import HollowTorusUnit, StruphyCallableMapping, get_eigenvalues

log_bounds = ((0, 1), (0, 1), (0, 1))
logical_domain = Cube('C', bounds1=log_bounds[0], bounds2=log_bounds[1], bounds3=log_bounds[2])

use_struphy_mapping = mapping_name in ['str_hollow_torus', 'hollow_desc', 'hollow_gvec']

if use_struphy_mapping:
    # see https://struphy-hub.github.io/struphy/sections/domains.html
    from struphy import domains as str_domains
    from struphy import equils  as str_equils

    log_cavity_radius = cavRad / minRad
    assert 0 <= log_cavity_radius < 1, f'log_cavity_radius should be in [0, 1) but got {log_cavity_radius}'

    if mapping_name == 'hollow_desc':
        str_domain = str_domains.DESCunit(str_equils.DESCequilibrium(rmin=log_cavity_radius, use_nfp=False))
    elif mapping_name == 'hollow_gvec':
        str_domain = str_domains.GVECunit(str_equils.GVECequilibrium(rmin=log_cavity_radius, use_nfp=False))
    else:
        str_domain = str_domains.HollowTorus(a1=cavRad, a2=minRad, R0=majRad, tor_period=1)

    # str_domain is already defined on the unit cube: no normalization needed
    struphy_callable_map = StruphyCallableMapping(
        struphy_domain=str_domain, logical_ldim=3, physical_pdim=3, normalize_factors=(1, 1, 1))

    # Spline interpolation of the Struphy mapping, then symbolic mapping whose callable is the spline
    map_discrete = SplineCallableMapping.from_mapping(
        struphy_callable_map, ncells=ncells, degree=degree, periodic=periodic, bounds=log_bounds)
    mapping = map_discrete.to_defined_mapping('M')
    domain  = mapping(logical_domain)

elif mapping_name == 'hollow_torus':
    mapping = HollowTorusUnit('HTU', a1=cavRad, a2=minRad, R0=majRad)
    domain  = mapping(logical_domain)

else:
    raise ValueError(f'Unknown mapping_name: {mapping_name}')

dim_normal_harmonic_space = 1  # one cavity

/Users/campos/Work/codes/pyccel/psydac_project/struphy/src/struphy/fields_background/equils.py:2409: UserWarning: self.units =<struphy.physics.physics.Units object at 0x342ea7770>, no rescaling performed in DESC output.
  warnings.warn(


### Discretize the domain and the de Rham sequence

When the symbolic domain carries a spline mapping, `discretize(domain)` builds the geometry from it
directly; otherwise the number of cells and the periodicity must be given.

In [21]:
from sympde.topology           import Derham
from psydac.api.discretization import discretize

derham = Derham(domain)

if use_struphy_mapping:
    domain_h = discretize(domain)
else:
    domain_h = discretize(domain, ncells=ncells, periodic=periodic)

derham_h = discretize(derham, domain_h, degree=degree)

V0,   V1,   V2,   V3   = derham.spaces
V0h,  V1h,  V2h,  V3h  = derham_h.spaces
V0cs, V1cs, V2cs, V3cs = [Vh.coeff_space for Vh in derham_h.spaces]

### Assemble the mass matrices and the discrete operators

We also build the Dirichlet projectors $DP_k$ (which set the boundary degrees of freedom to zero), the
mass matrices $M_0, M_1$ restricted to homogeneous boundary conditions, and CG solvers for them
preconditioned with the LST preconditioners.

In [22]:
from sympde.calculus import inner
from sympde.expr     import integral, BilinearForm
from sympde.topology import elements_of

from psydac.api.settings   import PSYDAC_BACKEND_GPYCCEL
from psydac.linalg.basic   import IdentityOperator
from psydac.linalg.solvers import inverse

backend = PSYDAC_BACKEND_GPYCCEL

u0, v0 = elements_of(V0, names='u0, v0')
u1, v1 = elements_of(V1, names='u1, v1')
u2, v2 = elements_of(V2, names='u2, v2')
u3, v3 = elements_of(V3, names='u3, v3')

m0 = BilinearForm((u0, v0), integral(domain, u0*v0))
m1 = BilinearForm((u1, v1), integral(domain, inner(u1, v1)))
m2 = BilinearForm((u2, v2), integral(domain, inner(u2, v2)))
m3 = BilinearForm((u3, v3), integral(domain, u3*v3))

t0 = time.time()
m0h = discretize(m0, domain_h, (V0h, V0h), backend=backend)
m1h = discretize(m1, domain_h, (V1h, V1h), backend=backend)
m2h = discretize(m2, domain_h, (V2h, V2h), backend=backend)
m3h = discretize(m3, domain_h, (V3h, V3h), backend=backend)
t_discretize = time.time() - t0

t0 = time.time()
M0 = m0h.assemble()
M1 = m1h.assemble()
M2 = m2h.assemble()
M3 = m3h.assemble()
t_assembly = time.time() - t0

G, C, D = derham_h.derivatives(kind='linop')   # gradient, curl, divergence

DP0, DP1, DP2, _ = derham_h.dirichlet_projectors(kind='linop')
I0, I1, I2       = [IdentityOperator(Vcs) for Vcs in [V0cs, V1cs, V2cs]]

M0_0 = DP0 @ M0 @ DP0 + (I0 - DP0)
M1_0 = DP1 @ M1 @ DP1 + (I1 - DP1)

t0 = time.time()
M0_0_pc, = derham_h.LST_preconditioners(M0=M0, hom_bc=True)
M1_0_pc, = derham_h.LST_preconditioners(M1=M1, hom_bc=True)
t_pc = time.time() - t0

inv_M0_0 = inverse(M0_0, 'CG', pc=M0_0_pc, maxiter=1000, tol=1e-15)
inv_M1_0 = inverse(M1_0, 'CG', pc=M1_0_pc, maxiter=1000, tol=1e-15)

print(f'Discretization : {t_discretize:.3g}s')
print(f'Assembly       : {t_assembly:.3g}s')
print(f'Preconditioner : {t_pc:.3g}s')

Discretization : 48.7s
Assembly       : 0.00625s
Preconditioner : 0.257s


## Step 2 : The discrete 1-Hodge-Laplacian

With homogeneous boundary conditions, the (weak form of the) 1-Hodge-Laplacian reads

$$
S = C^T M_2 C + M_1 G\, M_{0,0}^{-1} DP_0\, G^T M_1 ,
$$

i.e. $\widetilde{curl}_h\,curl - grad\,\widetilde{div}_h$ tested against $V^1_h$.
We build it both as a matrix-free PSYDAC operator (used by the iterative solvers) and as a SciPy
sparse matrix (used by the eigenvalue solver). For the latter, $M_{0,0}^{-1}$ is computed exactly with
a sparse inverse, which is only affordable on coarse grids.

In [23]:
from scipy.sparse.linalg import inv

# PSYDAC (matrix-free)
S   = C.T @ M2 @ C + M1 @ G @ inv_M0_0 @ DP0 @ G.T @ M1
S_0 = DP1 @ S @ DP1 + (I1 - DP1)

# SciPy (sparse)
t0 = time.time()
M0_s, M1_s, M2_s = M0.tosparse(), M1.tosparse(), M2.tosparse()
G_s,  C_s        = G.tosparse(),  C.tosparse()
DP0_s, DP1_s     = DP0.tosparse(), DP1.tosparse()
I0_s,  I1_s      = I0.tosparse(),  I1.tosparse()
t_sparse = time.time() - t0

M0_0_s = DP0_s @ M0_s @ DP0_s + (I0_s - DP0_s)
t0 = time.time()
inv_M0_0_s = inv(M0_0_s.tocsc())
inv_M0_0_s.eliminate_zeros()
t_inv = time.time() - t0

S_s   = C_s.T @ M2_s @ C_s + M1_s @ G_s @ inv_M0_0_s @ DP0_s @ G_s.T @ M1_s
S_0_s = DP1_s @ S_s @ DP1_s + (I1_s - DP1_s)

print(f'.tosparse()    : {t_sparse:.3g}s')
print(f'sparse inverse : {t_inv:.3g}s')

.tosparse()    : 0.0127s
sparse inverse : 0.0144s


## Step 3 : The normal harmonic field

The normal harmonic fields span the kernel of $S_0$. Since the hollow torus has one cavity, this
kernel is one-dimensional: we compute the few eigenvalues of $S_0$ closest to $0$ and take the
eigenvector of the smallest one.

In [24]:
from psydac.fem.basic        import FemField
from psydac.linalg.utilities import array_to_psydac

t0 = time.time()
eigenvalues, eigenvectors = get_eigenvalues(dim_normal_harmonic_space + 2, 1e-6, S_0_s, None)
t_evs = time.time() - t0
print(f'Eigenvalues    : {t_evs:.3g}s')

normal_harmonic_field = array_to_psydac(eigenvectors[:, 0], V1cs)
Normal_harmonic_field = FemField(V1h, normal_harmonic_field)

-----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  ----- 
computing 3 eigenvalues (and eigenvectors) close to sigma=1e-06 with scipy.sparse.eigsh...
done: eigenvalues found: array([-5.94079391e-12,  3.14124048e-02,  5.67981316e-02])
-----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----  -----

Eigenvalues    : 0.0636s


## Step 4 : The harmonic vector potentials

### Lifting potentials $A^b_{H,1}$ and $A^b_{H,2}$

Each lifting potential is a discrete 1-form whose coefficients are $1$ on a single layer of edges and $0$
elsewhere, chosen so that its curl "goes around" one of the two tunnels:

* $A^b_{H,1}$ : edges of the 3rd (toroidal) component on the inner boundary $x_1 = 0$ (the cavity wall),
  at toroidal index $0$, for all poloidal indices;
* $A^b_{H,2}$ : edges of the 2nd (poloidal) component on the outer boundary $x_1 = 1$,
  at poloidal angle $\approx \pi$, for all toroidal indices.

The indices below refer to the global numbering of the $V^1_h$ coefficients
(components stored one after the other, each in lexicographic order $(i_1, i_2, i_3)$).

In [25]:
nx, ny, nz = ncells
px, py, pz = degree

# number of coefficients per direction for the 2nd and 3rd components
# (non-periodic 1st direction: nx+px, periodic directions: ny, nz)
Nx, Ny, Nz = nx + px, ny, nz

def get_lifting_field_1():
    array = np.zeros(V1cs.dimension)
    ix, iz = 0, 0
    start  = (Nx-1)*Ny*Nz + Nx*Ny*Nz   # skip the 1st and 2nd components
    for iy in range(Ny):
        array[start + ix*Ny*Nz + iy*Nz + iz] = 1
    return array_to_psydac(array, V1cs)

def get_lifting_field_2():
    array = np.zeros(V1cs.dimension)
    iy    = int(np.ceil((Ny-1)/2))     # theta ~ pi
    ix    = Nx - 1                     # r = r_max
    start = (Nx-1)*Ny*Nz               # skip the 1st component
    for iz in range(Nz):
        array[start + ix*Ny*Nz + iy*Nz + iz] = 1
    return array_to_psydac(array, V1cs)

lifting_field_1 = get_lifting_field_1()
lifting_field_2 = get_lifting_field_2()

Lifting_field_1 = FemField(V1h, lifting_field_1)
Lifting_field_2 = FemField(V1h, lifting_field_2)

### Correction potentials $A^0_{H,1}$ and $A^0_{H,2}$

The correction potentials solve the Hodge-Laplace problems

$$
\left( \widetilde{curl}_h\ curl - grad\ \widetilde{div}_h \right) A^0_{H,i} = - \widetilde{curl}_h\ curl\, A^b_{H,i}, \qquad A^0_{H,i} \in H_0(curl).
$$

Since $S_0$ has a non-trivial kernel (the normal harmonic field), we solve on its orthogonal complement:
with $P_{\mathcal{H}}$ the orthogonal projection onto $\{\text{normal harmonic field}\}^\perp$ (in the
Euclidean inner product of coefficients), we use the invertible operator
$P_{\mathcal{H}} S_0 P_{\mathcal{H}} + (I - P_{\mathcal{H}})$ and project the right-hand side accordingly.

In [26]:
from psydac.linalg.basic import MatrixFreeLinearOperator

norm_squared_nhf = normal_harmonic_field.inner(normal_harmonic_field)
dot  = lambda x: x - (x.inner(normal_harmonic_field) / norm_squared_nhf) * normal_harmonic_field
P_H1 = MatrixFreeLinearOperator(domain=V1cs, codomain=V1cs, dot=dot, dot_transpose=dot)

S_0_kernel_free = P_H1 @ S_0 @ P_H1 + (I1 - P_H1)

maxiter = 20000
inv_S_0_kernel_free = inverse(S_0_kernel_free, 'CG', maxiter=maxiter, tol=cor_tol)

rhs1_0 = P_H1 @ DP1 @ (-C.T @ M2 @ C @ lifting_field_1)
rhs2_0 = P_H1 @ DP1 @ (-C.T @ M2 @ C @ lifting_field_2)

print(f'Maxiter = {maxiter} - Tol = {cor_tol}')

t0 = time.time()
correction_field1 = inv_S_0_kernel_free @ rhs1_0
print(f'Correction potential 1 obtained in {time.time()-t0:.3g}s : {inv_S_0_kernel_free.get_info()}')

t0 = time.time()
correction_field2 = inv_S_0_kernel_free @ rhs2_0
print(f'Correction potential 2 obtained in {time.time()-t0:.3g}s : {inv_S_0_kernel_free.get_info()}')

Maxiter = 20000 - Tol = 1e-12
Correction potential 1 obtained in 13.3s : {'niter': 5059, 'success': True, 'res_norm': 8.090065110488066e-13}
Correction potential 2 obtained in 13.2s : {'niter': 5287, 'success': True, 'res_norm': 4.488480084464545e-13}


### Harmonic vector potentials and tangent harmonic fields

$A_{H,i} = A^b_{H,i} + A^0_{H,i}$, and the tangent harmonic fields are $B_{H,i} = curl\,A_{H,i}$,
normalized in $L^2$.

In [27]:
harvepo1 = lifting_field_1 + correction_field1
harvepo2 = lifting_field_2 + correction_field2

Harvepo1 = FemField(V1h, harvepo1)
Harvepo2 = FemField(V1h, harvepo2)

tangent_harmonic_field1 = C @ harvepo1
tangent_harmonic_field2 = C @ harvepo2

tangent_harmonic_field1 /= np.sqrt(M2.dot_inner(tangent_harmonic_field1, tangent_harmonic_field1))
tangent_harmonic_field2 /= np.sqrt(M2.dot_inner(tangent_harmonic_field2, tangent_harmonic_field2))

Tangent_harmonic_field1 = FemField(V2h, tangent_harmonic_field1)
Tangent_harmonic_field2 = FemField(V2h, tangent_harmonic_field2)

## Step 5 : Verification

### The normal harmonic field

A normal harmonic field $A$ should satisfy $A \times n = 0$ on $\partial\Omega$ (which is the case for
$A = DP_1 A$), $curl\,A = 0$ and $\widetilde{div}_h A = 0$. The boundary here consists of the two
faces $x_1 = 0$ and $x_1 = 1$.

In [28]:
from sympde.topology import Union, NormalVector
from sympde.calculus import cross

boundary = Union(domain.get_boundary(0, -1), domain.get_boundary(0, 1))
nn       = NormalVector('nn')

weak_D = - inv_M0_0 @ DP0 @ G.T @ M1

normal_boundary_int   = BilinearForm((u1, v1), integral(boundary, inner(cross(nn, u1), cross(nn, v1))))
Normal_boundary_int   = discretize(normal_boundary_int, domain_h, (V1h, V1h), backend=backend).assemble()

a          = normal_harmonic_field
diff       = a - DP1 @ a
curl_a     = C @ a
weak_div_a = weak_D @ a

print(f'|| A x n        ||_L2(boundary) = {np.sqrt(Normal_boundary_int.dot_inner(a, a)):.3g}')
print(f'|| A - DP1(A)   ||_L2(domain)   = {np.sqrt(M1.dot_inner(diff, diff)):.3g}')
print(f'|| curl A       ||_L2(domain)   = {np.sqrt(M2.dot_inner(curl_a, curl_a)):.3g}')
print(f'|| weak-div A   ||_L2(domain)   = {np.sqrt(M0.dot_inner(weak_div_a, weak_div_a)):.3g}')

|| A x n        ||_L2(boundary) = 9.32e-16
|| A - DP1(A)   ||_L2(domain)   = 1.36e-21
|| curl A       ||_L2(domain)   = 1.31e-11
|| weak-div A   ||_L2(domain)   = 5.6e-13


### The tangent harmonic fields

A tangent harmonic field $B$ should satisfy $B \cdot n = 0$ on $\partial\Omega$ (which is the case for
$B = DP_2 B$), $div\,B = 0$ and $\widetilde{curl}_h B = 0$.

In [29]:
weak_C = inv_M1_0 @ DP1 @ C.T @ M2

tangent_boundary_int = BilinearForm((u2, v2), integral(boundary, inner(u2, nn) * inner(v2, nn)))
Tangent_boundary_int = discretize(tangent_boundary_int, domain_h, (V2h, V2h), backend=backend).assemble()

for i, b in enumerate((tangent_harmonic_field1, tangent_harmonic_field2), start=1):
    diff        = b - DP2 @ b
    div_b       = D @ b
    weak_curl_b = weak_C @ b
    print(f'Tangent harmonic field {i}:')
    print(f'  || B.n          ||_L2(boundary) = {np.sqrt(Tangent_boundary_int.dot_inner(b, b)):.3g}')
    print(f'  || B - DP2(B)   ||_L2(domain)   = {np.sqrt(M2.dot_inner(diff, diff)):.3g}')
    print(f'  || div B        ||_L2(domain)   = {np.sqrt(M3.dot_inner(div_b, div_b)):.3g}')
    print(f'  || weak-curl B  ||_L2(domain)   = {np.sqrt(M1.dot_inner(weak_curl_b, weak_curl_b)):.3g}')

Tangent harmonic field 1:
  || B.n          ||_L2(boundary) = 1.26e-16
  || B - DP2(B)   ||_L2(domain)   = 8.16e-36
  || div B        ||_L2(domain)   = 1.49e-16
  || weak-curl B  ||_L2(domain)   = 8.11e-12
Tangent harmonic field 2:
  || B.n          ||_L2(boundary) = 1e-16
  || B - DP2(B)   ||_L2(domain)   = 2.42e-36
  || div B        ||_L2(domain)   = 1.23e-17
  || weak-curl B  ||_L2(domain)   = 1.58e-12


## Step 6 : Save results

The fields are exported with the `OutputManager`, then evaluated on a refined grid and written to a
`.vtu` file by the `PostProcessManager` (for ParaView or PyVista).

In [30]:
from mpi4py import MPI
from psydac.api.postprocessing import OutputManager, PostProcessManager

comm = MPI.COMM_WORLD

save_fields = {
    'Normal_harmonic_field'   : Normal_harmonic_field,
    'Tangent_harmonic_field1' : Tangent_harmonic_field1,
    'Tangent_harmonic_field2' : Tangent_harmonic_field2,
    'Lifting_field_1'         : Lifting_field_1,
    'Lifting_field_2'         : Lifting_field_2,
    'Harvepo1'                : Harvepo1,
    'Harvepo2'                : Harvepo2,
}

if save:
    os.makedirs(output_dir, exist_ok=True)

    Om = OutputManager(f'{output_dir}/space_info.yml', f'{output_dir}/field_info.h5',
                       comm=comm, save_mpi_rank=True, mode='w')
    Om.add_spaces(V1=V1h)
    Om.add_spaces(V2=V2h)
    Om.export_space_info()
    Om.set_static()
    Om.export_fields(**save_fields)
    Om.close()

    Pm = PostProcessManager(domain=domain, space_file=f'{output_dir}/space_info.yml',
                            fields_file=f'{output_dir}/field_info.h5', comm=comm)
    Pm.export_to_vtk(f'{output_dir}/{params_name}', grid=None, npts_per_cell=[6, 6, 6],
                     fields=save_fields.keys())
    Pm.close()
    print(f'Fields written to {vtu_file}')

Fields written to hollow_torus_output/visu_hollow_desc_cavRad0.5_n4_4_4_d2_tol1e-12.static.vtu


## Step 7 : Visualization with PyVista

The domain is clipped by a vertical plane to show the inside. Colors show the field magnitude (colormap `cmap`, default `'turbo'`;
`'plasma'` is a perceptually uniform alternative) and black arrows the field direction at random points of the visible surface
(see the `plot_field` docstring for the arrow options). Requires `pyvista` (and, for interactive plots
in Jupyter, `trame`; otherwise use `pl.show(jupyter_backend='static')`).

The `ambient` argument lifts the shaded faces; pass `ambient=0` for PyVista's default lighting.

In [ ]:
import pyvista as pv

assert os.path.exists(vtu_file), f'{vtu_file} not found: run the cells above with save=True first.'
mesh = pv.read(vtu_file)

def plot_field(field_key, field_label, cmap='turbo', ambient=0.3,
               n_arrows=600, arrow_size=0.025, scale_arrows=False, arrow_color='black', seed=0):
    """
    Plot the magnitude of a vector field on the clipped domain, with arrows on its visible surface.

    arrow_size is the (maximal) arrow length as a fraction of the domain diagonal. With
    scale_arrows=False all arrows have the same length and only show the direction (the
    magnitude is given by the colors); with scale_arrows=True their length is proportional
    to the local magnitude.
    """
    field_data     = mesh.point_data[field_key]
    magnitude_name = field_key + '_magnitude'
    mesh[magnitude_name] = np.linalg.norm(field_data, axis=1)

    clipped = mesh.clip(normal=(-1, 0, 0), origin=(0.2, 0, 0), invert=False)

    pl = pv.Plotter(window_size=(1000, 850))
    pl.add_mesh(clipped, scalars=magnitude_name, cmap=cmap, ambient=ambient,
                show_scalar_bar=True, scalar_bar_args={'title': '|' + field_label + '|'})

    # Arrows at random cells of the visible surface (outer surface and cut plane), drawn with
    # probability proportional to the cell area so that they are evenly spread (the clip creates
    # many small cells along the cut); arrows rooted inside would be hidden by the opaque volume
    surface = clipped.extract_surface().point_data_to_cell_data().compute_cell_sizes(length=False, volume=False)
    areas   = surface.cell_data['Area']
    rng     = np.random.default_rng(seed)
    idx     = rng.choice(surface.n_cells, min(n_arrows, surface.n_cells), replace=False, p=areas / areas.sum())
    vecs    = surface.cell_data[field_key][idx]
    mags    = np.linalg.norm(vecs, axis=1)

    arrows = pv.PolyData(surface.cell_centers().points[idx])
    arrows['direction'] = vecs / (mags[:, None] + 1e-30)
    arrows['length']    = mags / (mesh[magnitude_name].max() + 1e-30) if scale_arrows else np.ones(len(idx))
    glyphs = arrows.glyph(orient='direction', scale='length', factor=arrow_size * mesh.length)
    pl.add_mesh(glyphs, color=arrow_color)

    pl.add_text(field_label, font_size=12)
    pl.set_position([1.5, -1.5, .5])
    pl.set_viewup([0, 0, 1])
    pl.show_bounds(grid='front', location='outer', all_edges=True)
    pl.show()

In [37]:
plot_field('Normal_harmonic_field', 'Normal Harmonic Field')

Widget(value='<iframe src="http://localhost:50477/index.html?ui=P_0x3aa632490_10&reconnect=auto" class="pyvist…

In [33]:
plot_field('Tangent_harmonic_field1', 'Tangent Harmonic Field 1')

Widget(value='<iframe src="http://localhost:50477/index.html?ui=P_0x32a78d950_6&reconnect=auto" class="pyvista…

In [34]:
plot_field('Tangent_harmonic_field2', 'Tangent Harmonic Field 2')

Widget(value='<iframe src="http://localhost:50477/index.html?ui=P_0x32a78d310_7&reconnect=auto" class="pyvista…

In [35]:
plot_field('Harvepo1', 'Harmonic Vector Potential 1')

Widget(value='<iframe src="http://localhost:50477/index.html?ui=P_0x32a78ead0_8&reconnect=auto" class="pyvista…

In [36]:
plot_field('Harvepo2', 'Harmonic Vector Potential 2')

Widget(value='<iframe src="http://localhost:50477/index.html?ui=P_0x32a78f250_9&reconnect=auto" class="pyvista…